In [3]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

tables = {
    "customer": pd.read_csv("DimCustomer.csv"),
    "account": pd.read_csv("DimAccount.csv"),
    "transaction": pd.read_csv("FactTransaction.csv"),
    "product": pd.read_csv("Dimproduct.csv"),
    "subcategory": pd.read_csv("DimProductSubCategory.csv"),
    "category": pd.read_csv("DimProductCategory.csv")
}

In [4]:
for name, df in tables.items():
    print(f"{name}: {df.shape}")

customer: (100, 8)
account: (193, 8)
transaction: (10000, 8)
product: (27, 3)
subcategory: (9, 3)
category: (3, 2)


In [8]:
for name, df in tables.items():
    print(f"\n{name.upper()}")
    print(df.columns.tolist())


CUSTOMER
['CustomerID', 'FullName', 'DOB', 'Gender', 'Region', 'Email', 'Status', 'JoinDate']

ACCOUNT
['AccountID', 'CustomerID', 'AccountType', 'OpenDate', 'ClosedDate', 'Status', 'RegistrationID', 'Balance']

TRANSACTION
['TransactionID', 'AccountID', 'TransactionDate', 'TransactionAmount', 'TransactionType', 'TransactionChannel', 'ProductID', 'Status']

PRODUCT
['ProductID', 'ProductSubcategoryID', 'ProductName']

SUBCATEGORY
['ProductSubCategoryID', 'ProductCategoryID', 'ProductSubCategoryName']

CATEGORY
['ProductCategoryID', 'ProductCategoryName']


In [9]:
# checking dtypes
for name, df in tables.items():
    print(f"\n{'='*50}")
    print(name.upper())
    print(f"{'='*50}")
    df.info()


CUSTOMER
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   CustomerID  100 non-null    int64 
 1   FullName    100 non-null    object
 2   DOB         100 non-null    object
 3   Gender      100 non-null    object
 4   Region      100 non-null    object
 5   Email       100 non-null    object
 6   Status      100 non-null    object
 7   JoinDate    100 non-null    object
dtypes: int64(1), object(7)
memory usage: 6.4+ KB

ACCOUNT
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 193 entries, 0 to 192
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   AccountID       193 non-null    int64  
 1   CustomerID      193 non-null    int64  
 2   AccountType     193 non-null    object 
 3   OpenDate        193 non-null    object 
 4   ClosedDate      107 non-null    object 
 5   Stat

In [10]:
# customer date 
tables["customer"]["DOB"] = pd.to_datetime(
    tables["customer"]["DOB"],
    format="%d/%m/%Y"
)

tables["customer"]["JoinDate"] = pd.to_datetime(
    tables["customer"]["JoinDate"],
    format="%d/%m/%Y"
)

In [11]:
# account date
tables["account"]["OpenDate"] = pd.to_datetime(
    tables["account"]["OpenDate"],
    format="%Y-%m-%d"
)

tables["account"]["ClosedDate"] = pd.to_datetime(
    tables["account"]["ClosedDate"],
    format="%Y-%m-%d",
    errors="coerce"
)

In [12]:
# transactionn date 
tables["transaction"]["TransactionDate"] = pd.to_datetime(
    tables["transaction"]["TransactionDate"],
    format="%m/%d/%Y"
)

In [13]:
# verification
for name, df in tables.items():
    print(f"\n{name}")
    print(df.dtypes)


customer
CustomerID             int64
FullName              object
DOB           datetime64[ns]
Gender                object
Region                object
Email                 object
Status                object
JoinDate      datetime64[ns]
dtype: object

account
AccountID                  int64
CustomerID                 int64
AccountType               object
OpenDate          datetime64[ns]
ClosedDate        datetime64[ns]
Status                    object
RegistrationID             int64
Balance                  float64
dtype: object

transaction
TransactionID                  int64
AccountID                      int64
TransactionDate       datetime64[ns]
TransactionAmount            float64
TransactionType               object
TransactionChannel            object
ProductID                      int64
Status                        object
dtype: object

product
ProductID                int64
ProductSubcategoryID     int64
ProductName             object
dtype: object

subcategory
Produ

In [14]:
# checking missing values
for name, df in tables.items():
    print(f"\n{name.upper()}")
    print(df.isnull().sum())


CUSTOMER
CustomerID    0
FullName      0
DOB           0
Gender        0
Region        0
Email         0
Status        0
JoinDate      0
dtype: int64

ACCOUNT
AccountID          0
CustomerID         0
AccountType        0
OpenDate           0
ClosedDate        86
Status             0
RegistrationID     0
Balance            0
dtype: int64

TRANSACTION
TransactionID         0
AccountID             0
TransactionDate       0
TransactionAmount     0
TransactionType       0
TransactionChannel    0
ProductID             0
Status                0
dtype: int64

PRODUCT
ProductID               0
ProductSubcategoryID    0
ProductName             0
dtype: int64

SUBCATEGORY
ProductSubCategoryID      0
ProductCategoryID         0
ProductSubCategoryName    0
dtype: int64

CATEGORY
ProductCategoryID      0
ProductCategoryName    0
dtype: int64


In [17]:
# checking closed ad open status
tables["account"].groupby("Status")["ClosedDate"].apply(
    lambda x: x.isna().sum()
)

Status
Closed     0
Open      86
Name: ClosedDate, dtype: int64

In [18]:
# duplicates
for name, df in tables.items():
    print(
        f"{name}: {df.duplicated().sum()} duplicate rows"
    )

customer: 0 duplicate rows
account: 0 duplicate rows
transaction: 0 duplicate rows
product: 0 duplicate rows
subcategory: 0 duplicate rows
category: 0 duplicate rows


In [19]:
# Primary key validation
primary_keys = {
    "customer": "CustomerID",
    "account": "AccountID",
    "transaction": "TransactionID",
    "product": "ProductID",
    "subcategory": "ProductSubCategoryID",
    "category": "ProductCategoryID"
}

for table, key in primary_keys.items():
    df = tables[table]

    print(
        f"{table}: "
        f"nulls={df[key].isna().sum()}, "
        f"duplicates={df[key].duplicated().sum()}"
    )

customer: nulls=0, duplicates=0
account: nulls=0, duplicates=0
transaction: nulls=0, duplicates=0
product: nulls=0, duplicates=0
subcategory: nulls=0, duplicates=0
category: nulls=0, duplicates=0


In [20]:
# Relationship validation
print(
    "Invalid Customer IDs:",
    (~tables["account"]["CustomerID"].isin(
        tables["customer"]["CustomerID"]
    )).sum()
)

print(
    "Invalid Account IDs:",
    (~tables["transaction"]["AccountID"].isin(
        tables["account"]["AccountID"]
    )).sum()
)

print(
    "Invalid Product IDs:",
    (~tables["transaction"]["ProductID"].isin(
        tables["product"]["ProductID"]
    )).sum()
)

Invalid Customer IDs: 0
Invalid Account IDs: 0
Invalid Product IDs: 0


In [22]:
# summary
for name, df in tables.items():
    print(f"\n{name.upper()}")
    print(df.describe(include="all"))


CUSTOMER
        CustomerID      FullName                  DOB Gender          Region  \
count   100.000000           100                  100    100             100   
unique         NaN           100                  NaN      2               8   
top            NaN  Norma Fisher                  NaN   Male   Massachusetts   
freq           NaN             1                  NaN     53              19   
mean     50.500000           NaN  1983-08-18 22:04:48    NaN             NaN   
min       1.000000           NaN  1960-01-12 00:00:00    NaN             NaN   
25%      25.750000           NaN  1971-06-12 00:00:00    NaN             NaN   
50%      50.500000           NaN  1983-01-01 12:00:00    NaN             NaN   
75%      75.250000           NaN  1996-08-20 06:00:00    NaN             NaN   
max     100.000000           NaN  2007-03-12 00:00:00    NaN             NaN   
std      29.011492           NaN                  NaN    NaN             NaN   

                  Email     S

In [23]:
# SQL Server Connection

from sqlalchemy import create_engine
from urllib.parse import quote_plus

server = r"SUFILAPTOP\SQLEXPRESS"
database = "banking"
driver = quote_plus("ODBC Driver 17 for SQL Server")

connection_string = (
    f"mssql+pyodbc://@{server}/{database}"
    f"?driver={driver}&trusted_connection=yes"
)

engine = create_engine(connection_string)

tables = {
    "customer": pd.read_csv("DimCustomer.csv"),
    "account": pd.read_csv("DimAccount.csv"),
    "transaction": pd.read_csv("FactTransaction.csv"),
    "product": pd.read_csv("DimProduct.csv"),
    "subcategory": pd.read_csv("DimProductSubCategory.csv"),
    "category": pd.read_csv("DimProductCategory.csv")
}

date_columns = {
    "customer": ["DOB", "JoinDate"],
    "account": ["OpenDate", "ClosedDate"],
    "transaction": ["TransactionDate"]
}

for table, columns in date_columns.items():
    for column in columns:
        if column in tables[table].columns:
            tables[table][column] = pd.to_datetime(
                tables[table][column],
                errors="coerce"
            )

tables["fact_transaction"] = tables.pop("transaction")

for table_name, df in tables.items():

    print(f"Uploading {table_name}...")

    df.to_sql(
        name=table_name,
        con=engine,
        if_exists="replace",
        index=False
    )

    print(f"{table_name} uploaded successfully.")

for table_name in tables.keys():

    query = f"SELECT COUNT(*) AS row_count FROM [{table_name}]"

    result = pd.read_sql(query, engine)

    print(
        f"{table_name}: "
        f"{result.loc[0, 'row_count']} rows"
    )

print("\nAll banking tables uploaded successfully!")

C:\Users\mirsu\AppData\Local\Temp\ipykernel_26248\4010253455.py:35: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  tables[table][column] = pd.to_datetime(


Uploading customer...


C:\Users\mirsu\anaconda3\Lib\site-packages\pandas\io\sql.py:1648: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


customer uploaded successfully.
Uploading account...
account uploaded successfully.
Uploading product...
product uploaded successfully.
Uploading subcategory...
subcategory uploaded successfully.
Uploading category...
category uploaded successfully.
Uploading fact_transaction...
fact_transaction uploaded successfully.
customer: 100 rows
account: 193 rows
product: 27 rows
subcategory: 9 rows
category: 3 rows
fact_transaction: 10000 rows

All banking tables uploaded successfully!
